In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
import warnings
import optuna
import gc
from sklearn.model_selection import StratifiedKFold
from pandas.errors import PerformanceWarning
from sklearn.metrics import roc_auc_score
from optuna.samplers import TPESampler
from itertools import combinations
from xgboost import XGBClassifier
from tqdm import tqdm

warnings.simplefilter(action="ignore", category=PerformanceWarning)
TARGET = 'y'
NUMS = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
CATS = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

c:\Users\arnov\miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("C:/Users/arnov/Desktop/Kaggle/Playground series/Binary Classification with a Bank Dataset/train.csv", index_col='id')
test = pd.read_csv('C:/Users/arnov/Desktop/Kaggle/Playground series/Binary Classification with a Bank Dataset/test.csv', index_col='id')
orig = pd.read_csv('C:/Users/arnov/Desktop/Kaggle/Playground series/Binary Classification with a Bank Dataset/bank-full.csv', delimiter=';')
orig['y'] = orig['y'].replace({'yes': 1, 'no': 0})

train[CATS] = train[CATS].astype('category')
test[CATS] = test[CATS].astype('category')
orig[CATS] = orig[CATS].astype('category')


C:\Users\arnov\AppData\Local\Temp\ipykernel_14908\1099930491.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orig['y'] = orig['y'].replace({'yes': 1, 'no': 0})


In [3]:
TE_columns = []

columns = NUMS + CATS

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)
        
        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]
        TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

100%|██████████| 120/120 [00:57<00:00,  2.10it/s]


In [4]:
def target_encode(train, valid, test, col, target=TARGET, kfold=5, smooth=20, agg='mean'):
    train['kfold'] = ((train.index) % kfold)
    col_name = '_'.join(col)
    train[f'TE_{agg.upper()}_' + col_name] = 0.
    for i in range(kfold):
        df_tmp = train[train['kfold'] != i]
        if agg == 'mean': mn = train[target].mean()
        elif agg == 'median': mn = train[target].median()
        elif agg == 'min': mn = train[target].min()
        elif agg == 'max': mn = train[target].max()
        elif agg == 'nunique': mn = 0
        df_tmp = df_tmp[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
        df_tmp.columns = col + [agg, 'count']
        if agg == 'nunique':
            df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
        else:
            df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
        df_tmp_m = train[col + ['kfold', f'TE_{agg.upper()}_' + col_name]].merge(df_tmp, how='left', left_on=col, right_on=col)
        df_tmp_m.loc[df_tmp_m['kfold'] == i, f'TE_{agg.upper()}_' + col_name] = df_tmp_m.loc[df_tmp_m['kfold'] == i, 'TE_tmp']
        train[f'TE_{agg.upper()}_' + col_name] = df_tmp_m[f'TE_{agg.upper()}_' + col_name].fillna(mn).values
        df_tmp = train[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
        if agg == 'mean': mn = train[target].mean()
        elif agg == 'median': mn = train[target].median()
        elif agg == 'min': mn = train[target].min()
        elif agg == 'max': mn = train[target].max()
        elif agg == 'nunique': mn = 0
        df_tmp.columns = col + [agg, 'count']
        if agg == 'nunique':
            df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
        else:
            df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
        df_tmp_m = valid[col].merge(df_tmp, how='left', left_on=col, right_on=col)
        valid[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
        valid[f'TE_{agg.upper()}_' + col_name] = valid[f'TE_{agg.upper()}_' + col_name].astype('float32')

        df_tmp_m = test[col].merge(df_tmp, how='left', left_on=col, right_on=col)
        test[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
        test[f'TE_{agg.upper()}_' + col_name] = test[f'TE_{agg.upper()}_' + col_name].astype('float32')

        train = train.drop('kfold', axis=1)
        train[f'TE_{agg.upper()}_' + col_name] = train[f'TE_{agg.upper()}_' + col_name].astype('float32')

        return (train, valid, test)

def count_encode(train, valid, test, col):
    counts = train[col].value_counts()

    train[f'CE_{col}'] = train[col].map(counts)
    valid[f'CE_{col}'] = valid[col].map(counts).fillna(0)
    test[f'CE_{col}'] = test[col].map(counts).fillna(0)
    return (train, valid, test)

In [5]:
oof = np.zeros(len(train))
pred = np.zeros(len(test))

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

for idx, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
    X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
    y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
    X_test = test.copy()

    X_train = pd.concat([X_train, orig[FEATURES]])
    y_train = pd.concat([y_train, orig[TARGET]])

    for col in tqdm(TE_columns):
        X_train, X_val, X_test = target_encode(pd.concat([X_train, y_train], axis=1), X_val, X_test, [col], smooth=10, agg='mean')
        #X_train, X_val, X_test = target_encode(X_train, X_val, X_test, [col], smooth=10, agg='median')
        X_train = X_train.drop(TARGET, axis=1)
        X_train, X_val, X_test = count_encode(X_train, X_val, X_test, col)
    
        X_train = X_train.drop(col, axis=1)
        X_val = X_val.drop(col, axis=1)

        X_test = X_test.drop(col, axis=1)

    """
    param_grid = {'colsample_bytree': 0.3413131357034518, 
                  'subsample': 0.8991973526634904, 
                  'reg_lambda': 4.063399595895267, 
                  'reg_alpha': 2.917406632455954, 
                  'max_depth': 8}

    model = XGBClassifier(**param_grid, 
                          n_estimators=10000,
                          objective='binary:logistic',
                          eval_metric='auc',
                          learning_rate=0.01,
                          early_stopping_rounds=200,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)"""
    
    parameters_xgboost={
            'n_estimators': 3397,
            'max_leaves': 51,
            'min_child_weight': 4.250658044094101,
            'learning_rate': 0.021715410571123632,
            'subsample': 0.770488707564487,
            'colsample_bylevel': 0.6300036801974633,
            'colsample_bytree': 0.649643715753366,
            'reg_alpha': 5.3137581033535985,
            'reg_lambda': 2.650413432169319,
            'max_depth': 0,
            'grow_policy': 'lossguide',
            'tree_method': 'hist',
            'enable_categorical':True,
            'device':'cuda',
            'n_jobs':-1
    }
    model = XGBClassifier(**parameters_xgboost)
    
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1]

    print(f'Fold {idx + 1}: {roc_auc_score(y_val, oof[val_idx])}')

    del model, X_train, X_val, y_train, y_val, X_test
    gc.collect()
pred /= 5
print(f'CV AUC: {roc_auc_score(train[TARGET], oof)}')

100%|██████████| 120/120 [03:04<00:00,  1.53s/it]


[0]	validation_0-logloss:0.37366
[100]	validation_0-logloss:0.16730
[200]	validation_0-logloss:0.14243
[300]	validation_0-logloss:0.13599
[400]	validation_0-logloss:0.13315
[500]	validation_0-logloss:0.13160
[600]	validation_0-logloss:0.13056
[700]	validation_0-logloss:0.12986
[800]	validation_0-logloss:0.12929
[900]	validation_0-logloss:0.12888
[1000]	validation_0-logloss:0.12849
[1100]	validation_0-logloss:0.12824
[1200]	validation_0-logloss:0.12805
[1300]	validation_0-logloss:0.12783
[1400]	validation_0-logloss:0.12767
[1500]	validation_0-logloss:0.12756
[1600]	validation_0-logloss:0.12742
[1700]	validation_0-logloss:0.12730
[1800]	validation_0-logloss:0.12725
[1900]	validation_0-logloss:0.12719
[2000]	validation_0-logloss:0.12714
[2100]	validation_0-logloss:0.12705
[2200]	validation_0-logloss:0.12701
[2300]	validation_0-logloss:0.12698
[2400]	validation_0-logloss:0.12692
[2500]	validation_0-logloss:0.12691
[2600]	validation_0-logloss:0.12690
[2700]	validation_0-logloss:0.12689
[280

100%|██████████| 120/120 [03:11<00:00,  1.59s/it]


[0]	validation_0-logloss:0.37367
[100]	validation_0-logloss:0.16874
[200]	validation_0-logloss:0.14508
[300]	validation_0-logloss:0.13850
[400]	validation_0-logloss:0.13568
[500]	validation_0-logloss:0.13408
[600]	validation_0-logloss:0.13300
[700]	validation_0-logloss:0.13224
[800]	validation_0-logloss:0.13165
[900]	validation_0-logloss:0.13119
[1000]	validation_0-logloss:0.13082
[1100]	validation_0-logloss:0.13056
[1200]	validation_0-logloss:0.13032
[1300]	validation_0-logloss:0.13014
[1400]	validation_0-logloss:0.13000
[1500]	validation_0-logloss:0.12985
[1600]	validation_0-logloss:0.12977
[1700]	validation_0-logloss:0.12967
[1800]	validation_0-logloss:0.12959
[1900]	validation_0-logloss:0.12951
[2000]	validation_0-logloss:0.12943
[2100]	validation_0-logloss:0.12940
[2200]	validation_0-logloss:0.12934
[2300]	validation_0-logloss:0.12931
[2400]	validation_0-logloss:0.12929
[2500]	validation_0-logloss:0.12929
[2600]	validation_0-logloss:0.12929
[2700]	validation_0-logloss:0.12927
[280

100%|██████████| 120/120 [03:42<00:00,  1.85s/it]


[0]	validation_0-logloss:0.37380
[100]	validation_0-logloss:0.16905
[200]	validation_0-logloss:0.14550
[300]	validation_0-logloss:0.13923
[400]	validation_0-logloss:0.13659
[500]	validation_0-logloss:0.13506
[600]	validation_0-logloss:0.13402
[700]	validation_0-logloss:0.13324
[800]	validation_0-logloss:0.13275
[900]	validation_0-logloss:0.13236
[1000]	validation_0-logloss:0.13202
[1100]	validation_0-logloss:0.13171
[1200]	validation_0-logloss:0.13150
[1300]	validation_0-logloss:0.13136
[1400]	validation_0-logloss:0.13119
[1500]	validation_0-logloss:0.13110
[1600]	validation_0-logloss:0.13096
[1700]	validation_0-logloss:0.13091
[1800]	validation_0-logloss:0.13082
[1900]	validation_0-logloss:0.13077
[2000]	validation_0-logloss:0.13074
[2100]	validation_0-logloss:0.13069
[2200]	validation_0-logloss:0.13067
[2300]	validation_0-logloss:0.13063
[2400]	validation_0-logloss:0.13063
[2500]	validation_0-logloss:0.13063
[2600]	validation_0-logloss:0.13063
[2700]	validation_0-logloss:0.13066
[280

100%|██████████| 120/120 [03:04<00:00,  1.54s/it]


[0]	validation_0-logloss:0.37366
[100]	validation_0-logloss:0.16771
[200]	validation_0-logloss:0.14316
[300]	validation_0-logloss:0.13664
[400]	validation_0-logloss:0.13394
[500]	validation_0-logloss:0.13242
[600]	validation_0-logloss:0.13143
[700]	validation_0-logloss:0.13075
[800]	validation_0-logloss:0.13023
[900]	validation_0-logloss:0.12977
[1000]	validation_0-logloss:0.12946
[1100]	validation_0-logloss:0.12922
[1200]	validation_0-logloss:0.12902
[1300]	validation_0-logloss:0.12888
[1400]	validation_0-logloss:0.12872
[1500]	validation_0-logloss:0.12857
[1600]	validation_0-logloss:0.12846
[1700]	validation_0-logloss:0.12836
[1800]	validation_0-logloss:0.12828
[1900]	validation_0-logloss:0.12819
[2000]	validation_0-logloss:0.12814
[2100]	validation_0-logloss:0.12811
[2200]	validation_0-logloss:0.12805
[2300]	validation_0-logloss:0.12803
[2400]	validation_0-logloss:0.12803
[2500]	validation_0-logloss:0.12801
[2600]	validation_0-logloss:0.12800
[2700]	validation_0-logloss:0.12799
[280

100%|██████████| 120/120 [03:37<00:00,  1.81s/it]


[0]	validation_0-logloss:0.37373
[100]	validation_0-logloss:0.16863
[200]	validation_0-logloss:0.14427
[300]	validation_0-logloss:0.13774
[400]	validation_0-logloss:0.13491
[500]	validation_0-logloss:0.13326
[600]	validation_0-logloss:0.13225
[700]	validation_0-logloss:0.13154
[800]	validation_0-logloss:0.13104
[900]	validation_0-logloss:0.13066
[1000]	validation_0-logloss:0.13034
[1100]	validation_0-logloss:0.13009
[1200]	validation_0-logloss:0.12989
[1300]	validation_0-logloss:0.12970
[1400]	validation_0-logloss:0.12956
[1500]	validation_0-logloss:0.12945
[1600]	validation_0-logloss:0.12934
[1700]	validation_0-logloss:0.12926
[1800]	validation_0-logloss:0.12919
[1900]	validation_0-logloss:0.12913
[2000]	validation_0-logloss:0.12910
[2100]	validation_0-logloss:0.12903
[2200]	validation_0-logloss:0.12898
[2300]	validation_0-logloss:0.12897
[2400]	validation_0-logloss:0.12895
[2500]	validation_0-logloss:0.12895
[2600]	validation_0-logloss:0.12893
[2700]	validation_0-logloss:0.12891
[280

In [6]:
submission = pd.read_csv('C:/Users/arnov/Desktop/Kaggle/Playground series/Binary Classification with a Bank Dataset/sample_submission.csv')
submission['y'] = pred
submission.to_csv('xgb.csv', index=False)
pd.DataFrame({'xgb_oof': oof}).to_csv('xgb_oof.csv', index=False)
pd.DataFrame({'xgb_pred': pred}).to_csv('xgb_pred.csv', index=False)